# Synthetic Neuro-Oncology Document Generation Pipeline

Steps 2-4: Document generation → Span resolution → Quality filtering

**Prerequisites:** 630 patient profiles already generated (Step 1) and stored in `profiles/batch_*.json`

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 1: Environment Setup
# ══════════════════════════════════════════════════════════════

# Mount Google Drive for checkpointing
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install -q vllm transformers pydantic rapidfuzz tqdm datasketch orjson

# If vLLM fails on your GPU, uncomment these for the transformers fallback:
# !pip install -q bitsandbytes accelerate

# Copy pipeline code from Drive (adjust path as needed)
import shutil, os

DRIVE_PROJECT = '/content/drive/MyDrive/synth_neuro_onco'
LOCAL_PROJECT = '/content/synth_neuro_onco'

if os.path.exists(DRIVE_PROJECT):
    if os.path.exists(LOCAL_PROJECT):
        shutil.rmtree(LOCAL_PROJECT)
    shutil.copytree(DRIVE_PROJECT, LOCAL_PROJECT)
    print(f'Copied project from {DRIVE_PROJECT}')
else:
    print(f'Project not found at {DRIVE_PROJECT}. Upload manually or clone from git.')

os.chdir(LOCAL_PROJECT)
import sys
sys.path.insert(0, LOCAL_PROJECT)

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 2: Configuration
# ══════════════════════════════════════════════════════════════

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

from config.settings import PipelineConfig

# Auto-detect GPU and select model
config = PipelineConfig.from_colab(
    drive_root='/content/drive/MyDrive/synth_neuro_onco',
)

# Override paths for Colab layout
from pathlib import Path
config.profiles_dir = Path(LOCAL_PROJECT) / 'profiles'
config.few_shot_dir = Path(LOCAL_PROJECT) / 'data' / 'few_shot'
config.output_dir = Path(LOCAL_PROJECT) / 'data'

print(f'Provider: {config.llm_provider}')
print(f'Model: {config.llm_model}')
print(f'Checkpoint dir: {config.checkpoint_dir}')

# Detect GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 3: Load & Validate Patient Profiles
# ══════════════════════════════════════════════════════════════

import json
from pathlib import Path
from collections import Counter

profiles = []
for path in sorted(config.profiles_dir.glob('batch_*.json')):
    with open(path, encoding='utf-8') as f:
        profiles.extend(json.load(f))

print(f'Loaded {len(profiles)} profiles from {len(list(config.profiles_dir.glob("batch_*.json")))} files')

# Distribution check
from profiles_validation import normalize_diag
cats = Counter(normalize_diag(p) for p in profiles)
for cat, count in cats.most_common():
    print(f'  {cat}: {count} ({count/len(profiles)*100:.1f}%)')

# Document type counts
doc_types = Counter()
for p in profiles:
    for dt in p.get('document_types', []):
        doc_types[dt] += 1
print(f'\nTotal documents to generate: {sum(doc_types.values())}')
for dt, count in doc_types.most_common():
    print(f'  {dt}: {count}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 4: Step 2 — Generate Documents with Annotations
# ══════════════════════════════════════════════════════════════

from pipeline.llm_client import LLMClient
from pipeline.checkpointing import CheckpointManager
from pipeline.step2_generate import generate_documents

# Initialise LLM (this loads the model into GPU memory)
print('Loading LLM...')
llm_client = LLMClient(config)
print('LLM ready.')

# Initialise checkpointing
checkpoint_mgr = CheckpointManager(config.checkpoint_dir)

# Generate!
raw_documents = generate_documents(
    profiles=profiles,
    config=config,
    llm_client=llm_client,
    checkpoint_mgr=checkpoint_mgr,
)

print(f'\nGenerated {len(raw_documents)} raw documents')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 5: Step 3 — Resolve Spans to Character Offsets
# ══════════════════════════════════════════════════════════════

from pipeline.step3_resolve import resolve_all_spans

resolved_documents, resolution_stats = resolve_all_spans(raw_documents, config)

print(f'\nResolution statistics:')
print(f'  Total annotations: {resolution_stats["total_annotations"]}')
print(f'  Exact match:       {resolution_stats["exact"]:.1%}')
print(f'  Case-insensitive:  {resolution_stats["case_insensitive"]:.1%}')
print(f'  Whitespace-norm:   {resolution_stats["whitespace_norm"]:.1%}')
print(f'  Fuzzy match:       {resolution_stats["fuzzy"]:.1%}')
print(f'  Unresolved:        {resolution_stats["unresolved"]:.1%}')
print(f'  Value fallback:    {resolution_stats["value_fallback"]:.1%}')
print(f'\n  Documents accepted: {resolution_stats["documents_accepted"]}')
print(f'  Documents rejected: {resolution_stats["documents_rejected"]}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 6: Step 4 — Quality Filtering
# ══════════════════════════════════════════════════════════════

from pipeline.step4_filter import run_filter_pipeline

# Build profiles lookup
profiles_lookup = {p['patient_id']: p for p in profiles}

final_documents, filter_report = run_filter_pipeline(
    resolved_documents, config, profiles_lookup,
)

print(f'\nFilter results:')
print(f'  Input:  {filter_report["input_count"]} documents')
print(f'  Output: {filter_report["output_count"]} documents')
print(f'  Pass rate: {filter_report["pass_rate"]:.1%}')
print(f'\nRejection breakdown:')
for reason, count in filter_report['rejection_reasons'].items():
    print(f'  {reason}: {count}')
if filter_report['review_flags']:
    print(f'\nReview flags:')
    for reason, count in filter_report['review_flags'].items():
        print(f'  {reason}: {count}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 7: Export Training-Ready JSONL
# ══════════════════════════════════════════════════════════════

import json
from pathlib import Path

# Save to local data directory
output_dir = config.final_dir
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'train.jsonl'

with open(output_path, 'w', encoding='utf-8') as f:
    for doc in final_documents:
        record = {
            'note_id': doc['note_id'],
            'note_text': doc['note_text'],
            'entities': doc['entities'],
        }
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

print(f'Saved {len(final_documents)} documents to {output_path}')

# Also copy to Drive for persistence
drive_output = Path(config.checkpoint_dir).parent / 'output' / 'train.jsonl'
drive_output.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(output_path, drive_output)
print(f'Backed up to {drive_output}')

# Quick sanity check
print(f'\n--- Sample document ---')
sample = final_documents[0]
print(f'note_id: {sample["note_id"]}')
print(f'text length: {len(sample["note_text"])} chars')
print(f'entities: {len(sample["entities"])}')
for ent in sample['entities'][:5]:
    snippet = sample['note_text'][ent['start']:ent['end']]
    print(f'  [{ent["start"]}:{ent["end"]}] {ent["label"]} = "{snippet}"')